# Phase 2: Laboratory Calibration and Final Outcome Construction

This notebook takes the ZIP produced by Phase 1 and:

1. Loads the 4,158-person analytic cohort.
2. Applies the published NHANES 1999–2000 serum creatinine calibration.
3. Standardizes cystatin C to the IFCC reference scale.
4. Recalculates creatinine- and cystatin C–based eGFR.
5. Recreates the discordance and hidden-CKD outcomes.
6. Compares results before and after calibration by NHANES cycle.
7. Saves a calibrated analytic dataset for predictor merging and machine learning.

Upload `Cystatin_C_Feasibility_Outputs.zip` when prompted.

In [ ]:
# Cell 1 — Imports and folders

import io
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

PROJECT_DIR = Path("/content/cystatin_c_phase2")
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_DIR = PROJECT_DIR / "outputs"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Environment prepared")

In [ ]:
# Cell 2 — Upload the Phase 1 ZIP

try:
    from google.colab import files

    uploaded = files.upload()

    expected_name = "Cystatin_C_Feasibility_Outputs.zip"

    if expected_name in uploaded:
        zip_path = INPUT_DIR / expected_name
        zip_path.write_bytes(uploaded[expected_name])
    else:
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]

        if len(zip_names) != 1:
            raise ValueError(
                "Please upload exactly one ZIP file: "
                "Cystatin_C_Feasibility_Outputs.zip"
            )

        zip_path = INPUT_DIR / zip_names[0]
        zip_path.write_bytes(uploaded[zip_names[0]])

    print("✅ Uploaded:", zip_path.name)

except ImportError:
    raise RuntimeError("Run this notebook in Google Colab.")

In [ ]:
# Cell 3 — Extract and load the analytic dataset

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(INPUT_DIR)

analytic_path = INPUT_DIR / "nhanes_cystatin_creatinine_analytic.csv"

if not analytic_path.exists():
    raise FileNotFoundError(
        "The ZIP does not contain nhanes_cystatin_creatinine_analytic.csv"
    )

df = pd.read_csv(analytic_path)

required = [
    "SEQN", "CYCLE", "RIDAGEYR", "RIAGENDR",
    "SSCYPC", "LBXSCR", "WTSCY4YR"
]

missing_required = [column for column in required if column not in df.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print("✅ Dataset loaded")
print("Rows:", f"{len(df):,}")
print("Columns:", f"{df.shape[1]:,}")
print("\nCycle counts:")
print(df["CYCLE"].value_counts().sort_index())

In [ ]:
# Cell 4 — Preserve the original laboratory values

df["CREATININE_REPORTED"] = pd.to_numeric(
    df["LBXSCR"], errors="coerce"
)

df["CYSTATIN_C_REPORTED"] = pd.to_numeric(
    df["SSCYPC"], errors="coerce"
)

df["AGE"] = pd.to_numeric(df["RIDAGEYR"], errors="coerce")
df["FEMALE"] = (pd.to_numeric(df["RIAGENDR"], errors="coerce") == 2).astype(int)

print(
    df[
        [
            "CREATININE_REPORTED",
            "CYSTATIN_C_REPORTED",
            "AGE",
            "FEMALE"
        ]
    ].describe().round(3)
)

## Calibration rules used

**Serum creatinine**

For NHANES 1999–2000:

\[
	ext{standardized creatinine}
=
0.147 + 1.013 	imes 	ext{reported creatinine}
\]

No correction is applied to NHANES 2001–2002.

**Cystatin C**

For NHANES 1999–2002, the recommended IFCC-traceable conversion is:

\[
	ext{IFCC cystatin C}
=
1.12 	imes (	ext{reported cystatin C} - 0.12)
\]

We retain both original and calibrated values so sensitivity analyses can compare them.

In [ ]:
# Cell 5 — Apply laboratory calibration

df["CREATININE_CALIBRATED"] = np.where(
    df["CYCLE"].eq("1999-2000"),
    0.147 + 1.013 * df["CREATININE_REPORTED"],
    df["CREATININE_REPORTED"]
)

df["CYSTATIN_C_IFCC"] = (
    1.12 * (df["CYSTATIN_C_REPORTED"] - 0.12)
)

# Guard against impossible values after transformation.
df.loc[df["CREATININE_CALIBRATED"] <= 0, "CREATININE_CALIBRATED"] = np.nan
df.loc[df["CYSTATIN_C_IFCC"] <= 0, "CYSTATIN_C_IFCC"] = np.nan

print("✅ Calibration applied")
print(
    df[
        [
            "CREATININE_REPORTED",
            "CREATININE_CALIBRATED",
            "CYSTATIN_C_REPORTED",
            "CYSTATIN_C_IFCC"
        ]
    ].describe().round(3)
)

In [ ]:
# Cell 6 — Adult eGFR equations

def egfr_creatinine_2021(creatinine, age, female):
    creatinine = pd.to_numeric(creatinine, errors="coerce")
    age = pd.to_numeric(age, errors="coerce")
    female = pd.Series(female, index=creatinine.index).fillna(0).astype(bool)

    kappa = np.where(female, 0.7, 0.9)
    alpha = np.where(female, -0.241, -0.302)
    sex_factor = np.where(female, 1.012, 1.0)

    ratio = creatinine / kappa

    result = (
        142
        * np.minimum(ratio, 1) ** alpha
        * np.maximum(ratio, 1) ** -1.200
        * 0.9938 ** age
        * sex_factor
    )

    invalid = (
        creatinine.isna()
        | age.isna()
        | (creatinine <= 0)
        | (age < 18)
    )

    return pd.Series(
        np.where(invalid, np.nan, result),
        index=creatinine.index
    )


def egfr_cystatin_2012(cystatin_c, age, female):
    cystatin_c = pd.to_numeric(cystatin_c, errors="coerce")
    age = pd.to_numeric(age, errors="coerce")
    female = pd.Series(female, index=cystatin_c.index).fillna(0).astype(bool)

    ratio = cystatin_c / 0.8
    sex_factor = np.where(female, 0.932, 1.0)

    result = (
        133
        * np.minimum(ratio, 1) ** -0.499
        * np.maximum(ratio, 1) ** -1.328
        * 0.996 ** age
        * sex_factor
    )

    invalid = (
        cystatin_c.isna()
        | age.isna()
        | (cystatin_c <= 0)
        | (age < 18)
    )

    return pd.Series(
        np.where(invalid, np.nan, result),
        index=cystatin_c.index
    )

In [ ]:
# Cell 7 — Calculate original and calibrated eGFR values

# Original values reproduce the Phase 1 approach.
df["EGFR_CR_ORIGINAL"] = egfr_creatinine_2021(
    df["CREATININE_REPORTED"], df["AGE"], df["FEMALE"]
)

df["EGFR_CYS_ORIGINAL"] = egfr_cystatin_2012(
    df["CYSTATIN_C_REPORTED"], df["AGE"], df["FEMALE"]
)

# Calibrated values are the preferred primary analysis.
df["EGFR_CR_CALIBRATED"] = egfr_creatinine_2021(
    df["CREATININE_CALIBRATED"], df["AGE"], df["FEMALE"]
)

df["EGFR_CYS_CALIBRATED"] = egfr_cystatin_2012(
    df["CYSTATIN_C_IFCC"], df["AGE"], df["FEMALE"]
)

print("✅ eGFR values recalculated")

In [ ]:
# Cell 8 — Create original and calibrated outcomes

def create_outcomes(data, cr_column, cys_column, suffix):
    data[f"RATIO_{suffix}"] = data[cys_column] / data[cr_column]

    data[f"DISCORDANCE_20_{suffix}"] = (
        data[cys_column] < 0.80 * data[cr_column]
    ).astype(int)

    data[f"DISCORDANCE_30_{suffix}"] = (
        data[cys_column] < 0.70 * data[cr_column]
    ).astype(int)

    data[f"DISCORDANCE_40_{suffix}"] = (
        data[cys_column] < 0.60 * data[cr_column]
    ).astype(int)

    data[f"HIDDEN_CKD_{suffix}"] = (
        (data[cr_column] >= 60)
        & (data[cys_column] < 60)
    ).astype(int)

    return data


df = create_outcomes(
    df,
    "EGFR_CR_ORIGINAL",
    "EGFR_CYS_ORIGINAL",
    "ORIGINAL"
)

df = create_outcomes(
    df,
    "EGFR_CR_CALIBRATED",
    "EGFR_CYS_CALIBRATED",
    "CALIBRATED"
)

print("✅ Outcomes created")

In [ ]:
# Cell 9 — Compare original and calibrated prevalence

outcome_pairs = [
    ("DISCORDANCE_20_ORIGINAL", "DISCORDANCE_20_CALIBRATED"),
    ("DISCORDANCE_30_ORIGINAL", "DISCORDANCE_30_CALIBRATED"),
    ("DISCORDANCE_40_ORIGINAL", "DISCORDANCE_40_CALIBRATED"),
    ("HIDDEN_CKD_ORIGINAL", "HIDDEN_CKD_CALIBRATED"),
]

comparison_rows = []

for original, calibrated in outcome_pairs:
    comparison_rows.append({
        "Outcome": original.replace("_ORIGINAL", ""),
        "Original cases": int(df[original].sum()),
        "Original prevalence (%)": 100 * df[original].mean(),
        "Calibrated cases": int(df[calibrated].sum()),
        "Calibrated prevalence (%)": 100 * df[calibrated].mean(),
    })

comparison = pd.DataFrame(comparison_rows)

print("=" * 75)
print("ORIGINAL VS CALIBRATED OUTCOMES")
print("=" * 75)
display(comparison.round(2))

In [ ]:
# Cell 10 — Primary outcome by cycle after calibration

cycle_comparison = (
    df.groupby("CYCLE")
    .agg(
        Eligible_N=("SEQN", "count"),
        Original_cases=("DISCORDANCE_30_ORIGINAL", "sum"),
        Original_prevalence=("DISCORDANCE_30_ORIGINAL", "mean"),
        Calibrated_cases=("DISCORDANCE_30_CALIBRATED", "sum"),
        Calibrated_prevalence=("DISCORDANCE_30_CALIBRATED", "mean"),
    )
    .reset_index()
)

cycle_comparison["Original prevalence (%)"] = (
    100 * cycle_comparison.pop("Original_prevalence")
)

cycle_comparison["Calibrated prevalence (%)"] = (
    100 * cycle_comparison.pop("Calibrated_prevalence")
)

print("=" * 75)
print("PRIMARY OUTCOME BY CYCLE")
print("=" * 75)
display(cycle_comparison.round(2))

In [ ]:
# Cell 11 — Weighted prevalence after calibration

def weighted_prevalence(data, outcome, weight="WTSCY4YR"):
    analysis = data[[outcome, weight]].dropna().copy()
    analysis = analysis.loc[analysis[weight] > 0]

    return np.average(
        analysis[outcome],
        weights=analysis[weight]
    )


weighted_rows = []

for outcome in [
    "DISCORDANCE_20_CALIBRATED",
    "DISCORDANCE_30_CALIBRATED",
    "DISCORDANCE_40_CALIBRATED",
    "HIDDEN_CKD_CALIBRATED"
]:
    weighted_rows.append({
        "Outcome": outcome.replace("_CALIBRATED", ""),
        "Weighted prevalence (%)":
            100 * weighted_prevalence(df, outcome)
    })

weighted_calibrated = pd.DataFrame(weighted_rows)

print("=" * 75)
print("CALIBRATED WEIGHTED PREVALENCE")
print("=" * 75)
display(weighted_calibrated.round(2))

In [ ]:
# Cell 12 — Visual comparison of eGFR ratio by cycle

for cycle in sorted(df["CYCLE"].dropna().unique()):
    subset = df.loc[df["CYCLE"] == cycle]

    plt.figure(figsize=(8, 5))
    plt.hist(
        subset["RATIO_CALIBRATED"].dropna(),
        bins=45
    )
    plt.axvline(0.70, linestyle="--")
    plt.xlabel("Calibrated eGFR cystatin C / calibrated eGFR creatinine")
    plt.ylabel("Participants")
    plt.title(f"Calibrated eGFR Ratio — NHANES {cycle}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 13 — Save calibrated outputs

preferred_columns = [
    "SEQN",
    "CYCLE",
    "WTSCY4YR",
    "SDMVPSU",
    "SDMVSTRA",
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH1",
    "CREATININE_REPORTED",
    "CREATININE_CALIBRATED",
    "CYSTATIN_C_REPORTED",
    "CYSTATIN_C_IFCC",
    "EGFR_CR_ORIGINAL",
    "EGFR_CYS_ORIGINAL",
    "EGFR_CR_CALIBRATED",
    "EGFR_CYS_CALIBRATED",
    "RATIO_ORIGINAL",
    "RATIO_CALIBRATED",
    "DISCORDANCE_20_CALIBRATED",
    "DISCORDANCE_30_CALIBRATED",
    "DISCORDANCE_40_CALIBRATED",
    "HIDDEN_CKD_CALIBRATED",
]

# Keep all existing variables, with key analysis fields first.
remaining_columns = [
    column for column in df.columns
    if column not in preferred_columns
]

df = df[preferred_columns + remaining_columns]

calibrated_csv = OUTPUT_DIR / "nhanes_calibrated_analytic.csv"
comparison_csv = OUTPUT_DIR / "calibration_outcome_comparison.csv"
cycle_csv = OUTPUT_DIR / "calibration_cycle_comparison.csv"
weighted_csv = OUTPUT_DIR / "calibrated_weighted_prevalence.csv"

df.to_csv(calibrated_csv, index=False)
comparison.to_csv(comparison_csv, index=False)
cycle_comparison.to_csv(cycle_csv, index=False)
weighted_calibrated.to_csv(weighted_csv, index=False)

print("✅ Saved calibrated analytic dataset")

In [ ]:
# Cell 14 — Download the calibrated output ZIP

output_zip = Path("/content/Cystatin_C_Phase2_Calibrated_Outputs.zip")

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in [
        calibrated_csv,
        comparison_csv,
        cycle_csv,
        weighted_csv
    ]:
        zf.write(file_path, arcname=file_path.name)

print("✅ Created:", output_zip)

from google.colab import files
files.download(str(output_zip))

## What to send back

Upload `Cystatin_C_Phase2_Calibrated_Outputs.zip`.

The most important output is the table from **Cell 10**, because it will show whether proper laboratory calibration reduces the large difference between the two survey cycles.